[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [NumPy, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/numpy-deep-dive.html)

# dtype


## What you will be able to do

Read an array's dtype, choose another with `dtype=` and `astype`, and recognize what happens to a
value that meets a type too small for it, whether it wraps around, loses its fraction or raises.

## The idea

### The problem

The readings so far are decimals, eight bytes each. A year at four stations is 280 kilobytes, which
is nothing, and ten years of hourly readings at a thousand stations is 700 megabytes, which is a
lot of memory for one notebook. Weather loggers have long saved space by storing a reading as a whole
number of tenths of a degree, -3.6 as -36, in two bytes instead of eight. NumPy lets you do the
same, and the array becomes a quarter of the size.

The catch is that a two-byte whole number can only hold values from -32,768 to 32,767. Square a
reading of 20.8 degrees stored as 208, and the answer, 43,264, does not fit. Python's own integers
grow to whatever size an answer needs, so you may never have met this, and NumPy does not stop to
tell you. The square comes back as a negative number, and the variance or the total you build from
it is wrong.

### What a dtype is

> An array's **dtype** is the one type that every value in it has, such as `int16`, `int64` or
> `float64`. It fixes how many bytes each value takes, the range of values it can hold, and what
> arithmetic does at the edges of that range.

### Why it works that way

A dtype's name gives its kind and its size in bits. `int64` is a whole number in 64 bits, eight
bytes, the type `np.arange` chooses. `int16` is two bytes, and `int8` one. A `u` in front, as in
`uint8`, means unsigned, with no negative values and twice the room above zero. `float64` is a
decimal with about 15 significant digits, and `float32` half the size with about 6. NumPy also has
`bool` for `True` and `False`, and a text type written like `<U6`, text of up to six characters.

Whole number arithmetic in a fixed number of bits wraps around. Add 1 to the largest `int8`, 127,
and you get the smallest, -128, the way a car's odometer rolls over to zero. Checking every result
for overflow would slow down every operation on every array, so NumPy does not check, and the
wrong value arrives with no warning.

NumPy 2 does check in one place. When a plain Python number meets an array, it takes the array's
type, and a number too big for that type raises `OverflowError` before any arithmetic happens. Two
arrays of the same small type still wrap. Changing a decimal into a whole number, with `astype`
or by storing it into a whole number array, drops the fraction toward zero, so 2.7 becomes 2.

### Where this shows up

Images are the commonest place. A pixel is a `uint8` from 0 to 255, and brightening a photo by
adding to it wraps the brightest pixels around to dark. A column of a table has a dtype, which the
**Pandas, Deep Dive** guide relies on, and a file read into an array gets one too. The
**Performance** notebook measures what a smaller type saves, and the **Random Numbers** notebook
chooses the dtype of the numbers it draws.

### What this notebook covers

- reading `dtype`, `itemsize` and `nbytes`
- the range of each whole number type, from `np.iinfo`
- choosing a dtype when an array is made, and converting with `astype`
- what `astype` does to a fraction, and how `np.round` rounds a half
- the dtype arithmetic chooses for its result
- wrapping at the edge of a type's range, and the check NumPy 2 adds for Python numbers

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read it,
and read the output underneath it. Everything from Setup onward is where you start running things.

```python
import numpy as np

tenths = np.array([118, 127], dtype=np.int8)
print(tenths + 1)
print(np.iinfo(np.int8).max)
```

```
[ 119 -128]
127
```

118 plus 1 is 119. 127 plus 1 should be 128, and `int8` cannot hold 128, so the value wrapped around
to -128. Nothing in the output warns that it happened.

## Setup

Three imports, and the year of readings as lists and as an array.

- `numpy`, imported as `np`, is the library, and the cell prints its version
- `math`, `datetime` and `timedelta` build the readings

`readings_list` holds four lists of 8,760 readings, one for each station in `STATIONS`, made by
`year_of_readings`, the formula the **Why NumPy** notebook explained. `readings` is the same values
as one `float64` array of shape (4, 8760), with `nan` for Svalbard's silent day.

In [1]:
import math
from datetime import datetime, timedelta

import numpy as np

STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}   # each station's mean for the year
START = datetime(2025, 1, 1)
HOURS = 365 * 24


def year_of_readings(station):
    """A reading for every hour of 2025 at one station, as the sqlite3 guide makes them.
    Svalbard was silent on 2 March, so those 24 hours are None."""
    i = list(STATIONS).index(station)
    row = []
    for n in range(HOURS):
        hour = START + timedelta(hours=n)
        if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
            row.append(None)
        else:
            season = -math.cos(2 * math.pi * (n - 400) / HOURS)
            day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
            wobble = ((n * 37 + i * 101) % 17 - 8) / 10
            row.append(round(STATIONS[station] + 9 * season + 3 * day + wobble, 1) + 0.0)
    return row


readings_list = [year_of_readings(station) for station in STATIONS]
readings = np.array(readings_list, dtype=float)       # None becomes nan

print("numpy", np.__version__, "|", len(readings_list), "stations,", len(readings_list[0]), "hours each |", readings.shape)

numpy 2.1.3 | 4 stations, 8760 hours each | (4, 8760)


## Worked examples

### Every array has one dtype

`dtype` names the type, `itemsize` gives the bytes one value takes, and `nbytes` the bytes the whole
array takes:

In [2]:
print(readings.dtype, readings.itemsize, readings.nbytes)
print(np.arange(5).dtype)
print(np.array([True, False]).dtype)
print(np.array(["Bergen", "Oslo"]).dtype)

float64 8 280320
int64
bool
<U6


35,040 decimals at eight bytes each is 280,320 bytes. `<U6` is text of up to six characters, the
length of "Bergen", and every value in that array takes the room of the longest.

### The range of each whole number type

`np.iinfo` gives the smallest and the largest value a whole number type can hold:

In [3]:
for kind in [np.int8, np.uint8, np.int16, np.int32, np.int64]:
    limits = np.iinfo(kind)
    print(f"{kind.__name__:6} {limits.min:>27,} {limits.max:>27,}")

int8                          -128                         127
uint8                            0                         255
int16                      -32,768                      32,767
int32               -2,147,483,648               2,147,483,647
int64   -9,223,372,036,854,775,808   9,223,372,036,854,775,807


Each extra byte multiplies the range by 256. `uint8` holds nothing below zero, and room for 255 above
it.

### Choosing a dtype when an array is made

Every constructor takes `dtype=`, with a NumPy type or with Python's `int` and `float`, which stand
for `int64` and `float64`:

In [4]:
print(np.zeros(3, dtype=np.int16), np.zeros(3, dtype=np.int16).dtype)
print(np.array([8.2, 7.9], dtype=np.float32).dtype)
print(np.ones(2, dtype=int).dtype)

[0 0 0] int16
float32
int64


### Converting with astype

`astype` makes a new array of another type. Bergen's readings become whole tenths of a degree by
multiplying by 10, rounding, and converting to `int16`:

In [5]:
tenths = np.round(readings[0] * 10).astype(np.int16)

print(readings[0, :6])
print(tenths[:6], tenths.dtype)
print(tenths.min(), tenths.max())
print(tenths.nbytes, "bytes against", readings[0].nbytes)
print(np.array_equal(tenths / 10, readings[0]))

[-3.6 -3.7 -3.7 -3.5 -3.1 -2.5]
[-36 -37 -37 -35 -31 -25] int16
-48 208
17520 bytes against 70080
True


The readings run from -4.8 to 20.8 degrees, -48 to 208 tenths, well inside `int16`'s range. The
array takes a quarter of the memory, and nothing was lost, since dividing by 10 gives back exactly
the readings it was made from.

### What astype does to a fraction

`astype(int)` drops the fraction toward zero, and does not round. `np.round` rounds, and a value
exactly halfway goes to the nearest even number:

In [6]:
print(np.array([2.7, -2.7, 0.5]).astype(int))
print(np.round(np.array([0.5, 1.5, 2.5])))
print(np.round(readings[3, :6]).astype(int))

[ 2 -2  0]
[0. 2. 2.]
[-7 -8 -8 -8 -8 -7]


2.7 and -2.7 both moved toward zero, to 2 and -2. 0.5 and 2.5 rounded down and 1.5 rounded up, all
three to the even neighbor, which is how NumPy keeps a long run of halves from drifting upward.
`np.round` before `astype(int)` gives Tromso's readings to the nearest degree.

### The dtype arithmetic chooses

When two types meet, the result takes a type that holds both. A whole number type with a decimal
gives a decimal, division always gives a decimal, and a Python whole number takes the array's type:

In [7]:
print((tenths + tenths).dtype)
print((tenths + readings[0]).dtype)
print((tenths * 1.5).dtype)
print((tenths / 10).dtype)
print((tenths + 1).dtype)
print(tenths.sum().dtype, tenths.sum())

int16
float64
float64
float64
int16
int64 700790


`tenths.sum()` is `int64`. NumPy adds up small whole number types in `int64`, because a total is the
result most likely to outgrow its type, so a sum is safe where an elementwise result may not be.

### Arithmetic wraps at the edge of a type's range

`uint8` holds the hours of a day comfortably. Multiplying by 60 for minutes since midnight keeps the
type, and the later hours run past 255:

In [8]:
hour_of_day = np.arange(24, dtype=np.uint8)
minutes = hour_of_day * 60

print(minutes.dtype)
print(minutes[:5])
print(minutes[-4:])
print(hour_of_day[-4:].astype(int) * 60)

uint8
[  0  60 120 180 240]
[176 236  40 100]
[1200 1260 1320 1380]


Hour 4 is 240 minutes, the last that fits. 20 times 60 is 1,200, and `uint8` kept only what was left
after taking away 256 four times, 176. Converting to `int` first, a type with room, gives the real
minutes.

### Memory for each choice

The year of readings in three types:

In [9]:
print("float64", readings.nbytes)
print("float32", readings.astype(np.float32).nbytes)
print("int16  ", np.round(readings[:2] * 10).astype(np.int16).nbytes * 2)

float64 280320
float32 140160
int16   70080


`int16` cannot hold `nan`, so the third line converts the two stations without missing hours and
doubles the bytes, to compare like with like. `float32` halves the memory and keeps `nan`, at the cost of digits. It holds about 6 significant digits, which is plenty for a thermometer that reads to a tenth.

## Your turn

Five tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/numpy-deep-dive/03-dtype-solutions.ipynb).

**1.** Print the dtype and itemsize of `np.arange(10)` and of `np.linspace(0, 1, 10)`.

In [10]:
# your code here

**2.** Print the largest value an `int32` can hold, using `np.iinfo`.

In [11]:
# your code here

**3.** Round Tromso's readings, row 3, to whole degrees, convert them to `int`, and print the first six.

In [12]:
# your code here

**4.** Make Oslo's readings, row 1, as `float32`, and print how many bytes they take against the `float64` row.

In [13]:
# your code here

**5.** Make `np.array([200], dtype=np.uint8)`, add an array holding 100 of the same type, and print the result. Then do the same in `int16` and compare.

In [14]:
# your code here

## Common errors

### No error, and squares that came out negative

In [15]:
squares = tenths ** 2

print(squares.dtype)
print(squares.min(), squares.max())
print((squares < 0).sum(), "negative squares")
print((tenths.astype(np.int64) ** 2).max())

int16
-32412 32761
529 negative squares
43264


A square is never negative, and 529 of Bergen's are. `tenths ** 2` stayed `int16`, and every
reading above 18.1 degrees, 181 tenths, has a square past 32,767, which wrapped. The real largest
square is 43,264. Convert to a type with room first, `tenths.astype(np.int64) ** 2`, or work in
`float64`, and check the range with `np.iinfo` whenever you choose a small type.

### OverflowError: Python integer -99999 out of bounds for int16

In [16]:
svalbard_tenths = np.zeros(HOURS, dtype=np.int16)

svalbard_tenths[1440:1464] = -99999                       # the silent day, marked as missing

OverflowError: Python integer -99999 out of bounds for int16

Some loggers mark a missing reading with a number no thermometer could show. -99999 is outside
`int16`'s range, and a Python number too big for the array's type raises in NumPy 2, before anything
is stored. NumPy 1 stored a wrapped value instead. Choose a marker inside the range, such as
-9999, or keep the readings as decimals and use `nan`.

### ValueError: cannot convert float NaN to integer

In [17]:
frost_hours = np.zeros(4, dtype=int)

frost_hours[2] = np.nan                                   # Svalbard, not counted yet

ValueError: cannot convert float NaN to integer

A whole number array has no `nan`, and storing one raises. Use a decimal array if a place may be
missing, `np.full(4, np.nan)`, or count with a marker such as -1 that the rest of the program
knows to skip.

### No error, and a mean stored without its fraction

In [18]:
means = np.zeros(4, dtype=int)
means[0] = np.nanmean(readings[0])

print(np.nanmean(readings[0]))
print(means)

7.999885844748858
[7 0 0 0]


`means` holds whole numbers, so Bergen's mean of 7.9998... was stored as 7, its fraction dropped
toward zero, with no warning. An array that will hold decimals needs a decimal dtype, which is
`np.zeros`'s own default: leave out `dtype=int`.

### UFuncTypeError: ufunc 'add' did not contain a loop with signature matching types (dtype('<U32'), dtype('int64')) -> None

In [19]:
mixed = np.array([8.2, "7.9", 7.5])                        # one reading arrived as text
print(mixed, mixed.dtype)

mixed + 1

['8.2' '7.9' '7.5'] <U32


UFuncTypeError: ufunc 'add' did not contain a loop with signature matching types (dtype('<U32'), dtype('int64')) -> None

One string in the list made the whole array text, `<U32`, and NumPy has no way to add a number to
text, which is what the message means by saying it has no loop for adding these two types. Convert
before doing arithmetic: `mixed.astype(float) + 1`. If a value is not a number at all, `astype` says
which.

## Recap

- Read an array's `dtype`, `itemsize` and `nbytes`, and a whole number type's range with `np.iinfo`
- Choose a type with `dtype=` or `astype`, and know that `astype(int)` drops a fraction toward zero
- Predict the type arithmetic gives, and that a sum of small whole numbers comes back as `int64`
- Spot a value that wrapped past its type's range, and convert to a type with room before the
  arithmetic
- Read NumPy 2's `OverflowError` as a Python number too big for the array's type

## What is next

The **Shape** notebook turns the (4, 8760) readings into days and hours and back, with `reshape`
and the `-1` that works out a dimension for you.

---

&#8592; **Previous:** [Creating Arrays](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/numpy-deep-dive/02-creating-arrays.ipynb)  &nbsp;·&nbsp;  [NumPy, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/numpy-deep-dive.html)
